# Cobain-Inspired Harmonic Ambiguity Generator

Exploring how power chords (no thirds = no declared tonality) create melodic freedom,
inspired by Kurt Cobain's songwriting style in Nirvana.

**Key concept**: Power chords are just root + fifth. Without a third, the chord declares
no major or minor quality. The melody above can imply *any* mode it wants — a different
tonal center over every chord. This is what gives Cobain songs like *Lithium* and *In Bloom*
their chromatic, shifting quality.

---
**Setup**: Make sure you're running this with the `musc448c` kernel (Python 3.11)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import ipywidgets as widgets
from IPython.display import display, Audio, clear_output, HTML

from cobain_generator.config import ProgressionConfig, MelodyConfig, MidiBuilderConfig, AudioConfig
from cobain_generator.chord_generator import PowerChordProgressionGenerator, BASE_INTERVAL_WEIGHTS, STYLE_BIAS
from cobain_generator.melody_generator import MelodyGenerator
from cobain_generator.midi_builder import MidiBuilder
from cobain_generator.audio_renderer import AudioRenderer
from cobain_generator.comparison import ComparisonGenerator, ComparisonConfig
from cobain_generator.scales import pitch_class_name, PITCH_CLASS_NAMES, SCALE_INTERVALS

os.makedirs('output/cobain', exist_ok=True)
os.makedirs('output/comparison', exist_ok=True)

renderer = AudioRenderer()
if renderer.is_available():
    print('Audio renderer ready -- WAV output enabled')
else:
    print('No FluidSynth found -- MIDI-only output (see assets/soundfonts/ README)')

print('All modules loaded OK')

## Part 1 — Chord Progression
Generate a Cobain-style power chord progression using weighted Markov chains over root intervals.

In [ ]:
NOTE_TO_PC = {
    'C':0,'C#':1,'Db':1,'D':2,'D#':3,'Eb':3,'E':4,'F':5,
    'F#':6,'Gb':6,'G':7,'G#':8,'Ab':8,'A':9,'A#':10,'Bb':10,'B':11
}

# --- Chord Progression Widgets ---
w_key   = widgets.Dropdown(options=list(NOTE_TO_PC.keys()), value='E', description='Root key:')
w_style = widgets.Dropdown(
    options=['grunge','verse','chorus','chromatic','ballad'],
    value='grunge', description='Style:'
)
w_bars  = widgets.IntSlider(value=8, min=2, max=32, step=2, description='Bars:')
w_tempo = widgets.IntSlider(value=120, min=60, max=200, step=5, description='Tempo (BPM):')
w_seed  = widgets.IntSlider(value=42, min=0, max=999, description='Seed:')
btn_gen = widgets.Button(description='Generate Progression', button_style='primary')
out_prog = widgets.Output()

# Store current progression for use in melody section
current_state = {'progression': None, 'prog_cfg': None}

def generate_and_display_progression(b=None):
    with out_prog:
        clear_output(wait=True)
        cfg = ProgressionConfig(
            root=NOTE_TO_PC[w_key.value],
            tempo_bpm=w_tempo.value,
            length_bars=w_bars.value,
            style=w_style.value,
            seed=w_seed.value,
        )
        gen = PowerChordProgressionGenerator(cfg)
        prog = gen.generate()
        current_state['progression'] = prog
        current_state['prog_cfg'] = cfg

        # Display chord sequence
        names = [pitch_class_name(r) + '5' for r, _ in prog]
        durs  = [d for _, d in prog]
        print(f'Progression ({len(prog)} chords, {sum(durs):.0f} beats):')
        print('  ' + '  '.join(names))
        print(f'  Durations: {[f"{d:.0f}b" for d in durs]}')

        # Interval analysis
        intervals = [(prog[i][0] - prog[i-1][0]) % 12 for i in range(1, len(prog))]
        inames = {0:'uni',1:'+m2',2:'+M2',3:'+m3',4:'+M3',5:'+P4',
                  6:'+tri',7:'+P5',8:'+m6',9:'+M6/bVI',10:'+m7/bVII',11:'-m2'}
        counts = {}
        for iv in intervals:
            counts[iv] = counts.get(iv, 0) + 1
        top = sorted(counts.items(), key=lambda x: -x[1])[:4]
        print(f'  Top moves: {"  ".join(f"{inames[iv]}(x{c})" for iv, c in top)}')

        # Visual: pitch-class wheel
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        # Left: chord timeline
        ax = axes[0]
        colors = plt.cm.Set3(np.linspace(0, 1, 12))
        x = 0
        for (root_pc, dur), name in zip(prog, names):
            ax.barh(0, dur, left=x, height=0.6,
                    color=colors[root_pc], edgecolor='black', linewidth=0.8)
            ax.text(x + dur/2, 0, name, ha='center', va='center', fontsize=9, fontweight='bold')
            x += dur
        ax.set_xlim(0, x)
        ax.set_ylim(-0.5, 0.5)
        ax.set_xlabel('Beats')
        ax.set_title(f'{w_key.value} {w_style.value.title()} -- Power Chord Timeline')
        ax.set_yticks([])

        # Right: interval distribution
        ax2 = axes[1]
        ivs = list(range(12))
        heights = [counts.get(iv, 0) for iv in ivs]
        labels = [inames[iv] for iv in ivs]
        bars = ax2.bar(ivs, heights, color='steelblue', edgecolor='black', linewidth=0.5)
        ax2.set_xticks(ivs)
        ax2.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
        ax2.set_ylabel('Count')
        ax2.set_title('Root Motion Intervals')

        plt.tight_layout()
        plt.show()

btn_gen.on_click(generate_and_display_progression)

display(widgets.VBox([
    widgets.HBox([w_key, w_style, w_tempo]),
    widgets.HBox([w_bars, w_seed]),
    btn_gen,
    out_prog,
]))
generate_and_display_progression()

## Part 2 — Melody Generation

Generate melody variations over the chord progression above.

**Per-chord modal color**: The scale mode is applied relative to each chord's root.
When the chord moves from E5 to G5, the active note pool shifts from E Dorian to G Dorian.
This constantly relocates the tonal center — the core of Cobain's melodic freedom.

**Hyperparameter guide**:
- `scale_mode` — which scale defines available notes (per chord root)
- `temperature` — how "risky" note choices are (low=predictable, high=adventurous)
- `note_density` — how many notes per beat (0.5=half notes, 2.0=eighth notes)
- `step_weight` — probability of moving by scale step vs leap
- `phrase_length` — beats before the melody resets to a chord anchor tone
- `rest_prob` — how often a note slot becomes silence

In [ ]:
SCALE_CHOICES = [
    'pentatonic_minor','pentatonic_major','dorian','mixolydian',
    'phrygian','aeolian','chromatic','blues','lydian','locrian'
]

# --- Melody Widgets ---
w_scale    = widgets.Dropdown(options=SCALE_CHOICES, value='pentatonic_minor', description='Scale mode:')
w_temp     = widgets.FloatSlider(value=1.0, min=0.1, max=2.5, step=0.1, description='Temperature:', readout_format='.1f')
w_density  = widgets.FloatSlider(value=1.0, min=0.25, max=4.0, step=0.25, description='Note density:', readout_format='.2f')
w_step     = widgets.FloatSlider(value=0.65, min=0.0, max=1.0, step=0.05, description='Step weight:', readout_format='.2f')
w_phrase   = widgets.FloatSlider(value=8.0, min=2.0, max=16.0, step=2.0, description='Phrase length:', readout_format='.0f')
w_rests    = widgets.FloatSlider(value=0.08, min=0.0, max=0.4, step=0.02, description='Rest prob:', readout_format='.2f')
w_oct_low  = widgets.IntSlider(value=4, min=2, max=6, description='Octave low:')
w_oct_high = widgets.IntSlider(value=5, min=3, max=7, description='Octave high:')
w_nvars    = widgets.IntSlider(value=3, min=1, max=8, description='Variations:')
w_mel_seed = widgets.IntSlider(value=42, min=0, max=999, description='Seed:')

btn_mel = widgets.Button(description='Generate Melodies', button_style='success')
out_mel = widgets.Output()

def generate_melodies(b=None):
    with out_mel:
        clear_output(wait=True)
        if current_state['progression'] is None:
            print('Generate a chord progression first (Part 1).')
            return

        prog = current_state['progression']
        prog_cfg = current_state['prog_cfg']

        mel_cfg = MelodyConfig(
            scale_mode=w_scale.value,
            temperature=w_temp.value,
            note_density=w_density.value,
            step_weight=w_step.value,
            phrase_length_beats=w_phrase.value,
            octave_low=w_oct_low.value,
            octave_high=w_oct_high.value,
            n_variations=w_nvars.value,
            rest_probability=w_rests.value,
        )
        # Pass seed through for reproducibility
        mel_cfg_dict = mel_cfg.__dict__.copy()

        import numpy as _np
        from cobain_generator.melody_generator import MarkovMelodyGenerator
        backend = MarkovMelodyGenerator()
        variations = []
        for i in range(w_nvars.value):
            rng = _np.random.default_rng(w_mel_seed.value + i * 1000)
            v = backend.generate_variation(prog, mel_cfg, rng)
            variations.append(v)

        builder = MidiBuilder(prog_cfg)
        tag = f"{w_key.value}_{w_style.value}_{w_scale.value}"

        print(f'Scale mode: {w_scale.value} (applied per chord root -- modal color shifts with each chord)')
        print(f'Generated {w_nvars.value} variation(s):\n')

        # Piano roll visualization
        fig, axes = plt.subplots(w_nvars.value, 1,
                                  figsize=(14, 2.5 * w_nvars.value),
                                  sharex=True)
        if w_nvars.value == 1:
            axes = [axes]

        chord_times = []
        t = 0
        for _, dur in prog:
            chord_times.append(t)
            t += dur
        chord_times.append(t)

        for i, (variation, ax) in enumerate(zip(variations, axes)):
            mid_path = f'output/cobain/{tag}_var{i+1:02d}.mid'
            wav_path = mid_path.replace('.mid', '.wav')
            builder.build(prog, variation, mid_path)

            # Piano roll
            beat = 0.0
            for (pitch, dur, vel) in variation:
                if pitch != -1:
                    color_intensity = vel / 127.0
                    ax.barh(pitch, dur, left=beat, height=0.8,
                            color=plt.cm.Blues(0.4 + 0.5 * color_intensity),
                            edgecolor='navy', linewidth=0.3)
                beat += dur

            # Chord boundary lines
            for ct in chord_times:
                ax.axvline(ct, color='red', alpha=0.3, linewidth=1)

            # Chord labels at top
            for j, ((root_pc, _), ct) in enumerate(zip(prog, chord_times[:-1])):
                dur = chord_times[j+1] - chord_times[j]
                ylim = ax.get_ylim()
                ax.text(ct + dur/2, ax.get_ylim()[1] if ylim[1] > 0 else 80,
                        pitch_class_name(root_pc)+'5',
                        ha='center', va='top', fontsize=7, color='red', alpha=0.7)

            ax.set_ylabel(f'Var {i+1}\nMIDI pitch')
            ax.set_xlim(0, sum(d for _, d in prog))

            if renderer.is_available():
                renderer.render(mid_path, wav_path)
                print(f'Variation {i+1}: {mid_path} | {wav_path}')
                display(Audio(wav_path))
            else:
                print(f'Variation {i+1}: {mid_path} (open in DAW/media player)')

        axes[-1].set_xlabel('Beats')
        fig.suptitle(f'{w_key.value} {w_style.value.title()} -- {w_scale.value} melody (per-chord modal color)', y=1.01)
        plt.tight_layout()
        plt.show()

btn_mel.on_click(generate_melodies)

display(widgets.VBox([
    widgets.HBox([w_scale, w_temp]),
    widgets.HBox([w_density, w_step]),
    widgets.HBox([w_phrase, w_rests]),
    widgets.HBox([w_oct_low, w_oct_high]),
    widgets.HBox([w_nvars, w_mel_seed]),
    btn_mel,
    out_mel,
]))

## Part 3 — Markov Transition Matrix
Visualize how the chord-style preset biases root motion probabilities.
Each cell shows P(move from root → root + N semitones).

In [ ]:
interval_names = ['0\n(uni)', '+m2', '+M2', '+m3\n(bIII)', '+M3', '+P4',
                  '+tri', '+P5', '+m6', '+M6\n(bVI)', '+m7\n(bVII)', '-m2']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
styles = ['grunge', 'verse', 'chorus', 'chromatic', 'ballad']
axes_flat = axes.flatten()

for ax, style in zip(axes_flat[:5], styles):
    weights = BASE_INTERVAL_WEIGHTS.copy()
    bias = STYLE_BIAS.get(style, {})
    for iv, mult in bias.items():
        weights[iv] *= mult
    if style == 'chromatic':
        weights = np.ones(12)
    weights /= weights.sum()

    # Make a 12x12 matrix (same probs from every root = transposition-invariant)
    mat = np.tile(weights, (12, 1))

    im = ax.imshow(mat, cmap='YlOrRd', aspect='auto', vmin=0, vmax=0.25)
    ax.set_xticks(range(12))
    ax.set_xticklabels(interval_names, fontsize=6)
    ax.set_yticks(range(12))
    ax.set_yticklabels(PITCH_CLASS_NAMES, fontsize=8)
    ax.set_xlabel('Interval to next chord')
    ax.set_ylabel('Current root')
    ax.set_title(f'Style: {style.title()}', fontweight='bold')

    # Annotate top-3 intervals
    top3 = np.argsort(weights)[-3:][::-1]
    for iv in top3:
        ax.axvline(iv, color='blue', alpha=0.25, linewidth=2)

# Bar chart comparison in last subplot
ax6 = axes_flat[5]
x = np.arange(12)
width = 0.15
colors_list = ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00']
for k, (style, color) in enumerate(zip(styles, colors_list)):
    weights = BASE_INTERVAL_WEIGHTS.copy()
    bias = STYLE_BIAS.get(style, {})
    for iv, mult in bias.items():
        weights[iv] *= mult
    if style == 'chromatic':
        weights = np.ones(12)
    weights /= weights.sum()
    ax6.bar(x + k * width, weights, width, label=style.title(), color=color, alpha=0.8)

ax6.set_xticks(x + width * 2)
ax6.set_xticklabels(interval_names, fontsize=6)
ax6.set_ylabel('Probability')
ax6.set_title('All Styles Compared')
ax6.legend(fontsize=7)

plt.suptitle('Cobain-Style Markov Chord Transition Probabilities\n(transposition-invariant: same probs from any root)',
             y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

print('Cobain fingerprints: +M6/bVI (9 semitones) = highest weight in Grunge -- Heart-Shaped Box, In Bloom')
print('                     +m7/bVII (10 semitones) = Smells Like Teen Spirit main riff')
print('                     +m3/bIII (3 semitones) = About a Girl, Come As You Are')

## Part 4 — Diatonic vs Power Chord Comparison

The same chord root motion, same melody — but one version uses full triads (major/minor thirds declared)
and the other uses power chords only. This directly demonstrates how harmonic ambiguity changes
the emotional quality of the music without touching the melody at all.

In [ ]:
btn_compare = widgets.Button(description='Run Comparison', button_style='warning')
out_compare = widgets.Output()

def run_comparison(b=None):
    with out_compare:
        clear_output(wait=True)
        if current_state['prog_cfg'] is None:
            print('Generate a chord progression first (Part 1).')
            return

        pcfg = current_state['prog_cfg']
        mel_cfg = MelodyConfig(
            scale_mode=w_scale.value,
            temperature=w_temp.value,
            note_density=w_density.value,
            step_weight=w_step.value,
            phrase_length_beats=w_phrase.value,
            octave_low=w_oct_low.value,
            octave_high=w_oct_high.value,
            rest_probability=w_rests.value,
        )
        cfg = ComparisonConfig(
            root=pcfg.root,
            tempo_bpm=pcfg.tempo_bpm,
            length_bars=pcfg.length_bars,
            style=pcfg.style,
            melody_config=mel_cfg,
            output_dir='output/comparison',
            seed=pcfg.seed if pcfg.seed is not None else 42,
        )
        results = ComparisonGenerator(cfg).run()
        print(results['summary'])
        print()

        if renderer.is_available():
            print('Power chords version:')
            display(Audio(results['power_chord_wav']))
            print('Diatonic triads version (same melody, same root motion):')
            display(Audio(results['diatonic_wav']))
        else:
            print(f'MIDI files written to output/comparison/')
            print('Open power_chords.mid and diatonic_chords.mid side by side in your DAW.')

btn_compare.on_click(run_comparison)
display(btn_compare, out_compare)

## Part 5 — Batch Generation

Generate a set of contrasting examples. Runs multiple style/scale
combinations and writes all MIDI files to `output/cobain/`.

In [ ]:
PRESENTATION_EXAMPLES = [
    # (key, style, scale, temperature, note_density, label)
    ('E', 'grunge',    'pentatonic_minor', 0.8, 1.0, 'Cobain Classic -- E grunge, pentatonic minor'),
    ('A', 'grunge',    'dorian',           1.0, 1.0, 'Dorian ambiguity -- A grunge, modal color shift'),
    ('E', 'verse',     'phrygian',         0.7, 0.5, 'Dark verse -- E verse, phrygian (half notes)'),
    ('E', 'chorus',    'mixolydian',       1.3, 2.0, 'Energetic chorus -- E chorus, mixolydian (eighth notes)'),
    ('D', 'chromatic', 'chromatic',        1.5, 1.5, 'In Bloom mode -- D chromatic scale + chromatic chords'),
    ('A', 'grunge',    'blues',            1.0, 1.5, 'Blues ambiguity -- A grunge, blues scale'),
]

btn_batch = widgets.Button(description='Generate All Presentation Examples', button_style='danger')
out_batch = widgets.Output()

def run_batch(b=None):
    with out_batch:
        clear_output(wait=True)
        print(f'Generating {len(PRESENTATION_EXAMPLES)} examples...\n')
        for key, style, scale, temp, density, label in PRESENTATION_EXAMPLES:
            print(f'  {label}')
            root = NOTE_TO_PC[key]
            pcfg = ProgressionConfig(root=root, tempo_bpm=120, length_bars=8,
                                      style=style, seed=42)
            mcfg = MelodyConfig(scale_mode=scale, temperature=temp,
                                 note_density=density, n_variations=1)
            gen = PowerChordProgressionGenerator(pcfg)
            prog = gen.generate()
            import numpy as _np
            rng = _np.random.default_rng(42)
            from cobain_generator.melody_generator import MarkovMelodyGenerator
            melody = MarkovMelodyGenerator().generate_variation(prog, mcfg, rng)
            tag = f'{key}_{style}_{scale}'.replace(' ', '_')
            mid_path = f'output/cobain/{tag}.mid'
            MidiBuilder(pcfg).build(prog, melody, mid_path)
            chords = '  '.join(pitch_class_name(r)+'5' for r, _ in prog)
            print(f'    Chords: {chords}')
            if renderer.is_available():
                renderer.render(mid_path, mid_path.replace('.mid', '.wav'))
                print(f'    -> {mid_path} + .wav')
            else:
                print(f'    -> {mid_path}')
        print('\nBatch complete. All files in output/cobain/')

btn_batch.on_click(run_batch)
display(btn_batch, out_batch)

---
## Musical Theory Notes (for presentation)

### Why power chords create harmonic ambiguity

A power chord is built from **two notes only**: the root and the perfect fifth (7 semitones above).
The interval that defines major vs. minor is the **third** (4 or 3 semitones above root).
Without a third, the chord is tonally uncommitted. Both `E major` and `E minor` share the
notes `E` and `B` — the power chord could be either.

### Kurt Cobain's approach

Cobain almost exclusively used power chords in his guitar parts. This gave him two freedoms:

1. **Chromatic chord motion**: Without thirds, any root can follow any other root without
   creating the dissonance a full triad would. In *In Bloom*, all 12 chromatic roots appear.

2. **Modal melody independence**: Because the chords declare no tonality, the melody doesn't
   have to respect one. In *Lithium*, the verse melody rapidly shifts between Dorian, Mixolydian,
   and minor depending on the phrasing — not the chords beneath.

### The Cobain Markov fingerprints

| Interval | Semitones | Cobain Example | Weight |
|----------|-----------|----------------|--------|
| +M6 (bVI down) | 9 | A5→F5 (Heart-Shaped Box), E5→C5 (In Bloom) | 16% |
| +P4 (IV up) | 5 | E5→A5 (About a Girl), A5→D5 (generic) | 14% |
| +m3 (bIII) | 3 | E5→G5 (About a Girl), A5→C5 (Come As You Are) | 12% |
| +m7 (bVII) | 10 | A5→G5 (Smells Like Teen Spirit) | 9% |

### Per-chord modal color (this generator)

In this tool, the `scale_mode` parameter is applied **relative to each chord's root**.
When the progression moves E5 → G5, the melody note pool shifts from E Dorian to G Dorian.
The melody can imply E minor in one bar and G major in the next, all over power chords
that never commit to either.